In [61]:
import numpy as np
import pandas as pd
import parselmouth
from parselmouth.praat import call
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [62]:
DATA_DIR = "H:\Voice Pathlogy Detection\Dataset\Saarbrucken_converted"
HEALTHY_DIR = os.path.join(DATA_DIR, "healthy")
PATHOLOGICAL_DIR = os.path.join(DATA_DIR, "pathological")

print(f"Data directories:")
print(f"  Healthy: {HEALTHY_DIR}")
print(f"  Pathological: {PATHOLOGICAL_DIR}")

Data directories:
  Healthy: H:\Voice Pathlogy Detection\Dataset\Saarbrucken_converted\healthy
  Pathological: H:\Voice Pathlogy Detection\Dataset\Saarbrucken_converted\pathological


In [63]:
def extract_mfcc_features(file_path, n_mfcc=13):
    try:
        #Load recording
        y, sr = librosa.load(file_path, sr=None) #sr is set to None to prevent downsampling to 22.05 kHz
        
        #Extract MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        delta_mfccs = librosa.feature.delta(mfccs)
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)

        #Calculate mean and std
        mfcc_mean = np.mean(mfccs, axis=1)
        mfcc_std = np.std(mfccs, axis=1)
        delta_mean = np.mean(delta_mfccs, axis=1)
        delta_std = np.std(delta_mfccs, axis=1)
        delta2_mean = np.mean(delta2_mfccs, axis=1)
        delta2_std = np.std(delta2_mfccs, axis=1)
        
        return np.concatenate([mfcc_mean, mfcc_std, delta_mean, delta_std, delta2_mean, delta2_std])
    
    except Exception as e:
        print(f"Error: {file_path}: {e}")
        return None

In [64]:
def extract_other_features(file_path):
    try:
        sound = parselmouth.Sound(file_path)
        pitch = call(sound, "To Pitch", 0.0, 75, 600)
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 600)
        
        # Jitter
        jitter_local = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        jitter_rap = call(point_process, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3)
        
        # Shimmer
        shimmer_local = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        shimmer_apq3 = call([sound, point_process], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        
        # HNR
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)
        
        return [jitter_local, jitter_rap, shimmer_local, shimmer_apq3, hnr]
    
    except Exception as e:
        print(f"Error: {file_path}: {e}")
        return None

In [65]:
def extract_all_features(file_path):
    mfcc = extract_mfcc_features(file_path)
    other = extract_other_features(file_path)
    
    return np.concatenate([mfcc, other])

## Loop

In [78]:
def extract_healthy():
    healthy_data = []
    patterns = ['a_h', 'a_n', 'a_l', 'a_lhl', 'i_h', 'i_n', 'i_l', 'i_lhl', 'u_h', 'u_n', 'u_l', 'u_lhl', 'iau']
    for person_id in tqdm(os.listdir(HEALTHY_DIR)):
        person_path = os.path.join(HEALTHY_DIR, person_id)

        #Sentence
        sentence_path = os.path.join(person_path, f"sentences\\{person_id}-phrase.wav")
        if os.path.exists(sentence_path):
            features = extract_all_features(sentence_path)
            healthy_data.append(np.concatenate([features, [0, 'healthy', person_id, 'phrase']]))

        #Vowels
        for pattern in patterns:
            vowel_path = os.path.join(person_path, f"vowels\\{person_id}-{pattern}.wav")
            features = extract_all_features(vowel_path)
            if os.path.exists(vowel_path):
                healthy_data.append(np.concatenate([features, [0, 'healthy', person_id, pattern]]))
                
    return healthy_data

In [79]:
mfcc_names = (
    [f"mfcc_{i}_mean" for i in range(1, 14)] +
    [f"mfcc_{i}_std" for i in range(1, 14)] +
    [f"d_mfcc_{i}_mean" for i in range(1, 14)] +
    [f"d_mfcc_{i}_std" for i in range(1, 14)] +
    [f"d2_mfcc_{i}_mean" for i in range(1, 14)] +
    [f"d2_mfcc_{i}_std" for i in range(1, 14)]
)
other_feature_names = ["jitter_local", "jitter_rap", "shimmer_local", "shimmer_apq3", "hnr"]
other_columns = ["label", "type", "person_id", "recording"]
column_names = np.concatenate([mfcc_names, other_feature_names, other_columns])

In [84]:
data = extract_healthy()
df = pd.DataFrame(data, columns=column_names)
df.to_csv("data/healthy.csv", index=False)

In [91]:
def extract_pathological():
    patterns = ['a_h', 'a_n', 'a_l', 'a_lhl', 'i_h', 'i_n', 'i_l', 'i_lhl', 'u_h', 'u_n', 'u_l', 'u_lhl', 'iau']
    pathologies = os.listdir(PATHOLOGICAL_DIR)
    len_pathologies = len(pathologies)
    n_pathology = 1
    for pathology in pathologies:
        print(f"Extracting for pathology ({n_pathology}/{len_pathologies})")
        n_pathology += 1
        pathological_data = []
        DIR = os.path.join(PATHOLOGICAL_DIR, pathology)
        for person_id in tqdm(os.listdir(DIR)):
            person_path = os.path.join(DIR, person_id)
    
            #Sentence
            sentence_path = os.path.join(person_path, f"sentences\\{person_id}-phrase.wav")
            if os.path.exists(sentence_path):
                features = extract_all_features(sentence_path)
                pathological_data.append(np.concatenate([features, [1, pathology, person_id, 'phrase']]))
    
            #Vowels
            for pattern in patterns:
                vowel_path = os.path.join(person_path, f"vowels\\{person_id}-{pattern}.wav")
                if os.path.exists(vowel_path):
                    features = extract_all_features(vowel_path)
                    pathological_data.append(np.concatenate([features, [1, pathology, person_id, pattern]]))

        #Save each pathology separately
        df_new = pd.DataFrame(pathological_data)
        df_new.to_csv('data/pathological.csv', mode='a', index=False, header=False)

In [96]:
#temp = []
#df = pd.DataFrame(temp, columns=column_names)
#df.to_csv("data/pathological.csv", index=False)

In [97]:
extract_pathological()

Extracting for pathology (1/71)


100%|██████████| 2/2 [00:05<00:00,  2.93s/it]


Extracting for pathology (2/71)


100%|██████████| 6/6 [00:21<00:00,  3.60s/it]


Extracting for pathology (3/71)


...

100%|██████████| 2/2 [00:10<00:00,  5.42s/it]


Extracting for pathology (70/71)


100%|██████████| 40/40 [02:35<00:00,  3.88s/it]


Extracting for pathology (71/71)


100%|██████████| 14/14 [01:04<00:00,  4.62s/it]
